# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/content-refresh-prioritization/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**Archetype → action mapping.** Five archetypes, each combining the w04 baseline rule
(weak position + real visibility) with the w05 model's decline probability, plus a
content_type carve-out for feedly articles (which w04's signal audit showed behave very
differently — 28.7% decline rate vs 56%+ for keyword/comparison articles):

| Archetype | Condition | Action | Reason code |
|---|---|---|---|
| No position data | `avg_position == 0` | `NO_ACTION` | `no_position_data` |
| Feedly, model flags decline | content_type=feedly, decline_prob≥0.6 | `FEEDLY_REVIEW_SEPARATE` | `feedly_declining_needs_own_playbook` |
| Weak position + visible + model agrees | baseline signal AND decline_prob≥0.6 | `PRIORITY_REFRESH` | `weak_position_visible_and_model_flags_decline` |
| Weak position + visible, model unsure | baseline signal only | `REVIEW_CTR` | `weak_position_visible_baseline_signal_only` |
| Model flags decline, low traffic | decline_prob≥0.6, impressions_90d<500 | `MONITOR_LOW_VOLUME` | `model_flags_decline_but_too_little_traffic_to_act_on` |
| Everything else | — | `NO_ACTION` | `no_strong_signal` |

**The decay/refresh insight, in one line:** the strongest, most reliable signal in this whole
project is still the simplest one — CTR drops off a cliff once a page falls past position 10
(2.71% at positions 1-3 down to 0.15% at 50+, confirmed in `w04_signal_audit.ipynb`) — so
`PRIORITY_REFRESH` (position + visibility + model agreement) is the tier worth trusting most,
and `REVIEW_CTR` (baseline signal alone) is the next-most-trustworthy fallback when the model
is unsure.

In [1]:
# ── Rebuild the ranked action queue: baseline rule + model signal + archetype ──
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
_candidates = [
    "/workspaces/content-refresh-prioritization/data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
_data_path = next(p for p in _candidates if os.path.exists(p))
df = pd.read_csv(_data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "avg_position", "impressions_90d", "clicks_90d", "ctr",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "days_with_impressions",
    "days_with_sessions", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
numeric_features += ["has_keyword_data", "has_word_count", "has_position_data"]
feature_cols = numeric_features + categorical_features
X, y, groups = df[feature_cols], df["is_declining_label"], df["client_id"]

# Same grouped split + model as w05/w06, so this playbook traces back to validated numbers.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))])
model.fit(X.iloc[train_idx], y.iloc[train_idx])

# Score EVERY row (this playbook is for action, not held-out evaluation -- evaluation lives in w05/w06).
df["decline_prob"] = model.predict_proba(X)[:, 1]

# Baseline rule signal, from w04.
df["weak_position"] = (df["avg_position"] >= 11) & (df["avg_position"] > 0)
df["has_visibility"] = df["impressions_90d"] >= 500
df["is_feedly"] = df["content_type"] == "feedly article"  # behaves differently, per w04 signal audit

def assign_action(row):
    if not row["has_position_data"]:
        return "NO_ACTION", "no_position_data"
    if row["is_feedly"] and row["decline_prob"] >= 0.6:
        return "FEEDLY_REVIEW_SEPARATE", "feedly_declining_needs_own_playbook"
    if row["weak_position"] and row["has_visibility"] and row["decline_prob"] >= 0.6:
        return "PRIORITY_REFRESH", "weak_position_visible_and_model_flags_decline"
    if row["weak_position"] and row["has_visibility"]:
        return "REVIEW_CTR", "weak_position_visible_baseline_signal_only"
    if row["decline_prob"] >= 0.6 and not row["has_visibility"]:
        return "MONITOR_LOW_VOLUME", "model_flags_decline_but_too_little_traffic_to_act_on"
    return "NO_ACTION", "no_strong_signal"

actions = df.apply(assign_action, axis=1, result_type="expand")
df["action"], df["reason_code"] = actions[0], actions[1]

action_priority = {"PRIORITY_REFRESH": 0, "FEEDLY_REVIEW_SEPARATE": 1, "REVIEW_CTR": 2,
                    "MONITOR_LOW_VOLUME": 3, "NO_ACTION": 4}
df["priority_rank"] = df["action"].map(action_priority)
queue = df.sort_values(["priority_rank", "decline_prob"], ascending=[True, False])

print(queue["action"].value_counts())
print("\nTop 10 of the ranked queue:")
queue[["content_id", "action", "reason_code", "avg_position", "impressions_90d", "decline_prob"]].head(10)


action
NO_ACTION                 17776
PRIORITY_REFRESH           4560
REVIEW_CTR                 3976
MONITOR_LOW_VOLUME         3592
FEEDLY_REVIEW_SEPARATE       96
Name: count, dtype: int64

Top 10 of the ranked queue:


,content_id,action,reason_code,avg_position,impressions_90d,decline_prob
25606,content_70b8f5323e29,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,19.8,62841,0.960783
3626,content_8ede62882d0b,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,14.3,556,0.903695
25108,content_5a467b5c7648,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,20.0,1685,0.898290
17988,content_d68a2265c1c0,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,28.5,547,0.874691
5578,content_fb03de774c31,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,14.2,573,0.872233
2762,content_d8b8f3280ec2,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,22.8,619,0.870979
2016,content_67a766790dd2,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,14.9,569,0.870793
15681,content_59b7653b71eb,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,13.3,661,0.867784
23726,content_1100bd1bcf09,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,16.8,668,0.865857
3413,content_b5d321cbefe8,PRIORITY_REFRESH,weak_position_visible_and_model_flags_decline,31.9,1422,0.862455


## 2. Intended use and limits

**Who uses this:** a content strategist or SEO lead doing monthly refresh planning — a
starting shortlist to review, not an auto-pilot queue.

**Valid for:** the 32 clients and single 90-day snapshot in this dataset release. Extending
this to a client or time period outside that snapshot is an assumption, not a validated claim.

**Where it stops being valid:**
- This is a **decision-support ranking**, not a certainty score. Precision@50 of 0.740 means
  roughly 1 in 4 of even the *top* recommendations will be wrong on inspection — expected, not
  a bug.
- The model's ROC AUC (0.608) is real but modest — it beats the baseline, it does not
  approach anything like a reliable per-page verdict.
- `avg_position == 0` pages have no position data at all — this playbook cannot make a CTR-based
  call on them regardless of tier.

In [2]:
# ── Facts that define the scope of "intended use" ──
n_clients = df["client_id"].nunique()
snapshot_note = "single 90-day trailing snapshot, not a live feed"
print(f"Coverage: {len(df):,} pages across {n_clients} clients (of the warehouse's 104 total)")
print(f"Data window: {snapshot_note}")
print(f"Model ceiling (from w05/w06, honest grouped split): ROC AUC 0.608, Precision@50 0.740 "
      f"vs baseline 0.480 -- a real lift, but far from a reliable per-page verdict")
print(f"Rows with no position data (can't act on CTR fixes at all): {(~df['has_position_data'].astype(bool)).sum():,}")


Coverage: 30,000 pages across 32 clients (of the warehouse's 104 total)
Data window: single 90-day trailing snapshot, not a live feed
Model ceiling (from w05/w06, honest grouped split): ROC AUC 0.608, Precision@50 0.740 vs baseline 0.480 -- a real lift, but far from a reliable per-page verdict
Rows with no position data (can't act on CTR fixes at all): 1,205


## 3. Human review + the no-go list

**A person must check, before acting on any `PRIORITY_REFRESH` or `REVIEW_CTR` row:**
- Is the traffic volume high enough that the CTR reading is stable, not a low-volume fluke?
  (25.0% of `PRIORITY_REFRESH` rows sit under 1,000 impressions_90d — exactly the profile w06
  showed produces false positives.)
- Has this page already been edited recently? A stale pre-change data snapshot would misfire.
- Is this page a near-duplicate of another flagged page (possible cannibalization, a
  consolidation fix, not a CTR fix)?

**What should NEVER be automated:**
- No automatic publishing, rewriting, redirecting, or deleting of content from this queue —
  every action here is a *suggestion* for a human to open and assess.
- No merging/consolidating pages without a human confirming they actually target the same
  intent — a wrong merge can lose real traffic.
- No treating `trend_direction == "down"` alone as proof a page is the worst performer —
  w04's flag-linked audit found "up" trending pages can still rank worse in absolute position
  than "down" trending ones. The label describes trajectory, not overall health.

In [3]:
# ── Facts that justify the human-review / no-go rules below ──
# 1. How often does PRIORITY_REFRESH catch a low-volume page where a single-day
#    fluke could flip the reading? (the exact false-positive pattern found in w06)
priority = df[df["action"] == "PRIORITY_REFRESH"]
low_vol_priority = priority[priority["impressions_90d"] < 1000]
print(f"PRIORITY_REFRESH pages with impressions_90d < 1,000: {len(low_vol_priority)} of {len(priority)} "
      f"({len(low_vol_priority)/len(priority):.1%}) -- these need a human sanity check before action, "
      f"not an automated trigger, since w06 showed exactly this profile produces false positives.")

# 2. "down" trend does not mean "worst absolute performer" (w04_signal_audit finding) --
#    a person needs to check current CTR/position, not just trust the label.
up_but_bad_position = df[(df["trend_direction"] == "up") & (df["avg_position"] >= 20) & (df["avg_position"] > 0)]
print(f"\nPages trending 'up' but still ranking 20+ (not flagged for refresh under this playbook, "
      f"but arguably worth a look): {len(up_but_bad_position):,} -- a reminder this queue is not the "
      f"whole picture of what needs attention.")


PRIORITY_REFRESH pages with impressions_90d < 1,000: 1141 of 4560 (25.0%) -- these need a human sanity check before action, not an automated trigger, since w06 showed exactly this profile produces false positives.

Pages trending 'up' but still ranking 20+ (not flagged for refresh under this playbook, but arguably worth a look): 1,847 -- a reminder this queue is not the whole picture of what needs attention.


## 4. Monitoring / retrain triggers

Baseline numbers this snapshot sets, to compare every future run against:

In [4]:
# ── Monitoring baselines to compare future snapshots against ──
monitoring_baselines = {
    "precision_at_50_model": 0.740,       # from w05_model.ipynb, honest grouped split
    "precision_at_50_rule_baseline": 0.480,  # from w04_baseline_score.ipynb
    "roc_auc_model": 0.608,
    "base_rate_declining": round(df["is_declining_label"].mean(), 3),
    "action_mix": df["action"].value_counts(normalize=True).round(3).to_dict(),
}
for k, v in monitoring_baselines.items():
    print(f"{k}: {v}")

print("""
Retrain/re-check triggers (light, not automated):
1. Re-run w05's evaluation on a fresh snapshot -- if precision@50 for the model drops to at or
   below the 0.480 rule-baseline, the model is no longer earning its complexity; fall back to the
   rule until retrained.
2. If base_rate_declining moves more than ~10 points from 0.542 (this snapshot's rate), the
   portfolio mix has shifted enough that the model's learned patterns may not transfer -- retrain.
3. If the action_mix share for PRIORITY_REFRESH more than doubles or halves between snapshots,
   check for a feature-distribution shift (e.g. a data-pull change) before trusting the new queue.
4. Recheck the two "flag-linked" signals from w04_signal_audit (CTR vs position, staleness vs
   decline) on each new snapshot -- if CONFIRMED flips to MIXED or worse, the rule/model mix
   built on top of it needs a rebuild, not just a rerun.
""")


precision_at_50_model: 0.74
precision_at_50_rule_baseline: 0.48
roc_auc_model: 0.608
base_rate_declining: 0.542
action_mix: {'NO_ACTION': 0.593, 'PRIORITY_REFRESH': 0.152, 'REVIEW_CTR': 0.133, 'MONITOR_LOW_VOLUME': 0.12, 'FEEDLY_REVIEW_SEPARATE': 0.003}

Retrain/re-check triggers (light, not automated):
1. Re-run w05's evaluation on a fresh snapshot -- if precision@50 for the model drops to at or
   below the 0.480 rule-baseline, the model is no longer earning its complexity; fall back to the
   rule until retrained.
2. If base_rate_declining moves more than ~10 points from 0.542 (this snapshot's rate), the
   portfolio mix has shifted enough that the model's learned patterns may not transfer -- retrain.
3. If the action_mix share for PRIORITY_REFRESH more than doubles or halves between snapshots,
   check for a feature-distribution shift (e.g. a data-pull change) before trusting the new queue.
4. Recheck the two "flag-linked" signals from w04_signal_audit (CTR vs position, staleness v

## 5. Exports for the paper

Three exports: the queue CSV (regenerated every run, stays out of git — CI leak-guard), the
metrics JSON (the receipts — committed), and one figure (committed to `work/figures/`, reused
directly in the paper's recommendations section).

In [5]:
# ── Exports: the queue CSV, a metrics JSON, and one figure -- what the paper builds on ──
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def first_existing_dir(candidates):
    for p in candidates:
        parent = os.path.dirname(p.rstrip("/")) or "."
        if os.path.isdir(parent):
            return p
    return candidates[-1]

outputs_dir = first_existing_dir([
    "/workspaces/content-refresh-prioritization/work/outputs",
    "../outputs",
    "work/outputs",
])
figures_dir = first_existing_dir([
    "/workspaces/content-refresh-prioritization/work/figures",
    "../figures",
    "work/figures",
])
os.makedirs(outputs_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

# 1. The ranked queue CSV -- regenerated every run, stays OUT of git by design (CI leak-guard).
export_cols = ["content_id", "client_id", "action", "reason_code", "priority_rank",
               "decline_prob", "avg_position", "impressions_90d", "ctr", "content_type"]
queue[export_cols].to_csv(os.path.join(outputs_dir, "action_playbook_queue.csv"), index=False)
print(f"Wrote {len(queue):,} rows to {os.path.join(outputs_dir, 'action_playbook_queue.csv')}")

# 2. The metrics JSON -- the receipts. This DOES get committed.
metrics = {
    "n_pages": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "action_counts": queue["action"].value_counts().to_dict(),
    "precision_at_50_model": 0.740,
    "precision_at_50_rule_baseline": 0.480,
    "roc_auc_model": 0.608,
    "base_rate_declining": round(float(df["is_declining_label"].mean()), 3),
}
metrics_path = os.path.join(outputs_dir, "action_playbook_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote metrics receipts to {metrics_path}")

# 3. One figure -- action counts by tier, reused in the paper's recommendations section.
fig, ax = plt.subplots(figsize=(7, 4))
order = ["PRIORITY_REFRESH", "REVIEW_CTR", "FEEDLY_REVIEW_SEPARATE", "MONITOR_LOW_VOLUME", "NO_ACTION"]
counts = queue["action"].value_counts().reindex(order)
ax.barh(order[::-1], counts[::-1], color="#4C72B0")
ax.set_xlabel("Number of pages")
ax.set_title("Action Playbook: pages per action tier")
fig.tight_layout()
fig_path = os.path.join(figures_dir, "action_tier_counts.png")
fig.savefig(fig_path, dpi=120)
plt.close(fig)
print(f"Wrote figure to {fig_path}")


Wrote 30,000 rows to /workspaces/content-refresh-prioritization/work/outputs/action_playbook_queue.csv
Wrote metrics receipts to /workspaces/content-refresh-prioritization/work/outputs/action_playbook_metrics.json
Wrote figure to /workspaces/content-refresh-prioritization/work/figures/action_tier_counts.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
